# Training Loop and Loss Functions

> The model is built, but all parameters are random numbers. How do we turn it from "guessing blindly" to "making predictions"? The answer is training -- but what exactly happens at each step of the training loop?
>
> This section uses a tiny example to break down the training loop: how data is organized, how loss is computed, how gradients flow, and how Chat Templates affect which positions contribute to loss. Every number will be computed by hand.

LLM training is essentially a "next token prediction" task: show the model preceding tokens, have it guess the next one, and compute loss when it guesses wrong. The core training loop is: tokenize the corpus into token sequences, construct input and label pairs (label is input shifted right by one position), run the model forward to get logits, compute cross-entropy loss, backpropagate, and update parameters.

This process looks simple, but there are several easily overlooked details: loss is computed across all token positions simultaneously (not sequentially), padding positions must be masked out of the loss, and for conversational data, loss is typically computed only on assistant responses.

## 1. The Simplest Training Example

Suppose we have a tiny model with a vocabulary of only 5 words, and we want to train it to predict the next token.

```
Vocabulary: [BOS=0, I=1, love=2, you=3, EOS=4]

Training data is a single sentence: "I love you"
token IDs: [BOS, I, love, you, EOS]
          = [0, 1, 2, 3, 4]
```

**Training goal**: given preceding tokens, predict the next one.

```
Given [BOS]           -> predict I
Given [BOS, I]        -> predict love
Given [BOS, I, love]  -> predict you
Given [BOS, I, love, you] -> predict EOS
```

These look like 4 independent prediction tasks, but Transformer has a magic property: **it can make predictions at all positions in parallel!**

## 2. Training Data and Labels

This is the most critical concept to understand. Look at this diagram:

```
Training sentence: [BOS,  I,   love, you,  EOS]
                    [0,   1,    2,    3,    4]

          +-------------------------------------+
Input:    |  BOS  |  I    | love  |  you  |
          |  [0]  | [1]   | [2]   | [3]   |
          +-------------------------------------+
                    |  model forward
          +-------------------------------------+
Model:    | logits | logits| logits| logits|
output:   |  [0]   |  [1]  |  [2]  |  [3]  |
          +-------------------------------------+
                    |  each position predicts next token
          +-------------------------------------+
Expected: |  I     | love  |  you  |  EOS  |
labels:   |  [1]   | [2]   |  [3]  |  [4]  |
          +-------------------------------------+

Input  = sentence with last token removed
Labels = sentence with first token removed (shifted right by one)
```

**This is called teacher forcing**: instead of using the model's own predictions to continue, we feed it the **correct answers**.

Why do this? Because it allows all positions to be trained **in parallel**, without waiting for previous positions' results.

In [ ]:
# Demonstrate this concept with a hand calculation
import torch

sentence = torch.tensor([0, 1, 2, 3, 4])  # [BOS, I, love, you, EOS]

print("Full sentence:", sentence.tolist())
print()

# Input: remove the last token
input_ids = sentence[:-1]  # [0, 1, 2, 3]
print("Input (last token removed):", input_ids.tolist())
print("Meaning:                        [BOS,  I,   love, you]")
print()

# Labels: remove the first token (shift right by one)
target_ids = sentence[1:]   # [1, 2, 3, 4]
print("Labels (first token removed):", target_ids.tolist())
print("Meaning:                        [I,   love, you,  EOS]")
print()

print("Position-by-position mapping:")
for i in range(len(input_ids)):
    print(f"  Position {i}: sees [{', '.join(str(x) for x in input_ids[:i+1].tolist())}] -> predicts {target_ids[i].item()}")

## 3. Cross-Entropy Loss

The model outputs a set of logits at each position (one score per word in the vocabulary). We need to compare them against the labels.

We use **Cross-Entropy Loss**:

```
For each position:
1. Convert logits to probabilities: softmax(logits)
2. Look up the probability assigned to the correct label
3. Compute -log(that probability)
4. Average across all positions
```

**Intuition**: If the correct label has probability 1.0 -> loss = -log(1.0) = 0 (perfect)
             If the correct label has probability 0.01 -> loss = -log(0.01) = 4.6 (terrible)

Let's walk through this step by step in code.

In [ ]:
# Simulate model output
import torch

vocab_size = 5
seq_len = 4  # input length

# Assume these are the model's output logits (batch=1)
# In practice these come from the model; here we create them manually
torch.manual_seed(123)
logits = torch.randn(1, seq_len, vocab_size)  # [batch=1, seq_len=4, vocab=5]
targets = torch.tensor([[1, 2, 3, 4]])          # [batch=1, seq_len=4]

print(f"Model output logits shape: {logits.shape}")
print(f"Labels shape: {targets.shape}")
print()

# Look at position 0's logits and label
print(f"Position 0 logits: {logits[0, 0].tolist()}")
print(f"Position 0 label:  {targets[0, 0].item()}")
print(f"-> The model must guess which of 5 words is word {targets[0, 0].item()}")

In [ ]:
# Compute loss by hand (understand where each number comes from)
import torch.nn.functional as F
import math

print("=== Hand-computed Cross-Entropy Loss ===")
print()

total_loss = 0.0
for pos in range(seq_len):
    # Logits at this position (vocab_size scores)
    pos_logits = logits[0, pos]  # [vocab_size]
    # Correct answer at this position
    correct_id = targets[0, pos].item()

    # Step 1: softmax to get probabilities
    probs = F.softmax(pos_logits, dim=-1)

    # Step 2: probability of the correct answer
    correct_prob = probs[correct_id].item()

    # Step 3: loss = -log(probability)
    pos_loss = -math.log(correct_prob)
    total_loss += pos_loss

    print(f"Position {pos}: correct=word{correct_id}, probability={correct_prob:.4f}, loss={pos_loss:.4f}")

# Step 4: average
manual_loss = total_loss / seq_len
print(f"\nAverage loss across all positions: {manual_loss:.4f}")

# Compare with PyTorch's built-in cross_entropy
pt_loss = F.cross_entropy(
    logits.reshape(-1, vocab_size),  # [batch*seq_len, vocab]
    targets.reshape(-1)               # [batch*seq_len]
).item()
print(f"PyTorch cross_entropy: {pt_loss:.4f}")
print(f"Match? {'Yes' if abs(manual_loss - pt_loss) < 1e-4 else 'No'}")

## 4. Token-Level or Sentence-Level?

**Answer: Token-level training.**

But note: **all tokens are trained in parallel simultaneously**, not one token at a time.

```
+----------------------------------------------+
|        One forward + backward pass            |
|                                              |
|  Loss = loss(pos 0) + loss(pos 1) + ...      |
|                                              |
|  Position 0 predicts token 1                  |
|  Position 1 predicts token 2  } all parallel  |
|  Position 2 predicts token 3                  |
|  Position 3 predicts token 4                  |
|                                              |
|  Gradient = dLoss/dW is the sum of           |
|  gradients from all positions                 |
|  Parameter update <- includes learning        |
|  signals from all positions                   |
+----------------------------------------------+
```

**Why not sentence-level?**
- Sentence-level means only predicting at one position (the end) -> signal is too sparse
- For a 100-token sentence, sentence-level gives only 1 supervision signal
- Token-level gives 100 supervision signals, 100x more efficient

**But it's also not "sequential token training."** All positions are computed in parallel during a single forward pass.
This is the key reason Transformers are faster than RNNs.

## 5. Training a Batch of Multiple Sentences

In real training, we don't process one sentence at a time. We process a batch (e.g., 32 sentences) packed into a single matrix.

```
Batch input:
[[BOS,  I,   love, you,  EOS,  PAD,  PAD],   <- Sentence 1 (5 valid tokens)
 [BOS,  hello, world, EOS, PAD,  PAD,  PAD]]   <- Sentence 2 (4 valid tokens)

Shape: [batch_size=2, seq_len=7]
```

Loss computation: average across all sentences and all positions (excluding PAD).

```python
loss = cross_entropy(logits.reshape(-1, vocab), targets.reshape(-1), ignore_index=PAD_ID)
#                                                    ^ ignore padding positions
```

In [ ]:
# Demonstrate batch training loss computation
import torch
import torch.nn.functional as F

PAD_ID = 0  # assume 0 is PAD

batch_input = torch.tensor([
    [0, 1, 2, 3, 4, 0, 0],  # [BOS, I, love, you, EOS, PAD, PAD]
    [0, 2, 4, 0, 0, 0, 0],  # [BOS, love, EOS, PAD, PAD, PAD, PAD]
])

# Labels = input shifted right by one
batch_target = torch.tensor([
    [1, 2, 3, 4, 0, 0, 0],  # [I, love, you, EOS, PAD, PAD, PAD]
    [2, 4, 0, 0, 0, 0, 0],  # [love, EOS, PAD, PAD, PAD, PAD, PAD]
])

print("Batch input:")
print(batch_input)
print()
print("Batch labels:")
print(batch_target)
print()

# Simulate model output
batch_logits = torch.randn(2, 7, 5)  # [batch=2, seq=7, vocab=5]

# Key: ignore_index=PAD_ID excludes PAD positions from loss
loss_with_ignore = F.cross_entropy(
    batch_logits.reshape(-1, 5),       # [14, 5]
    batch_target.reshape(-1),           # [14]
    ignore_index=PAD_ID
)

loss_without_ignore = F.cross_entropy(
    batch_logits.reshape(-1, 5),
    batch_target.reshape(-1)
)

print(f"Loss ignoring PAD: {loss_with_ignore.item():.4f}")
print(f"Loss including PAD: {loss_without_ignore.item():.4f}")
print(f"\nThe difference is large! PAD positions have meaningless predictions and should not contribute to loss.")

## 6. The Complete Training Loop

Now we combine the loss function, gradient computation, and optimizer into a complete training loop. In a real scenario, the training loop goes through the full cycle: data loading, tokenization, forward pass, loss computation, backpropagation, and parameter update.

But we haven't trained a real tokenizer yet, so we'll use a simplified setup:

- A tiny "pseudo-vocabulary" (a few dozen token IDs) to simulate tokenized input
- Manually construct the input-label offset (label = input shifted right by one, standard practice for autoregressive language models)
- Walk through the complete forward -> loss -> backward -> update flow

Although this uses simplified data, the structure is identical to real training. Understanding this means switching to a real tokenizer and data is just a matter of changing the input.

In [ ]:
# Reuse the MiniGPT from Part 4, simplified version
import torch
import torch.nn as nn
import math

def get_sinusoidal_encoding(seq_len, d_model):
    position = torch.arange(seq_len).unsqueeze(1)
    div_term = torch.exp(
        torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model)
    )
    pe = torch.zeros(seq_len, d_model)
    pe[:, 0::2] = torch.sin(position * div_term)
    pe[:, 1::2] = torch.cos(position * div_term)
    return pe

class MiniGPT(nn.Module):
    def __init__(self, vocab_size, d_model=64, num_heads=4, num_layers=4, max_seq_len=128):
        super().__init__()
        self.d_model = d_model
        self.vocab_size = vocab_size
        self.max_seq_len = max_seq_len

        self.token_emb = nn.Embedding(vocab_size, d_model)
        pe = get_sinusoidal_encoding(max_seq_len, d_model)
        self.register_buffer('pe', pe)

        # Simplified: not using ModuleList, just write two blocks directly
        self.attn1 = nn.MultiheadAttention(d_model, num_heads, batch_first=True)
        self.ffn1 = nn.Sequential(
            nn.Linear(d_model, 4*d_model), nn.ReLU(), nn.Linear(4*d_model, d_model)
        )
        self.norm1a = nn.LayerNorm(d_model)
        self.norm1f = nn.LayerNorm(d_model)

        self.attn2 = nn.MultiheadAttention(d_model, num_heads, batch_first=True)
        self.ffn2 = nn.Sequential(
            nn.Linear(d_model, 4*d_model), nn.ReLU(), nn.Linear(4*d_model, d_model)
        )
        self.norm2a = nn.LayerNorm(d_model)
        self.norm2f = nn.LayerNorm(d_model)

        self.ln_final = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size)

    def forward(self, x):
        batch_size, seq_len = x.shape
        x = self.token_emb(x) + self.pe[:seq_len, :]

        mask = torch.triu(torch.ones(seq_len, seq_len, device=x.device) * float('-inf'), diagonal=1)

        # Block 1
        attn_out, _ = self.attn1(x, x, x, attn_mask=mask)
        x = self.norm1a(x + attn_out)
        x = self.norm1f(x + self.ffn1(x))

        # Block 2
        attn_out, _ = self.attn2(x, x, x, attn_mask=mask)
        x = self.norm2a(x + attn_out)
        x = self.norm2f(x + self.ffn2(x))

        x = self.ln_final(x)
        return self.lm_head(x)

print("MiniGPT model definition complete!")

In [ ]:
# === Complete training loop demonstration ===

# 1. Prepare fake data (simulating tokenized text)
import torch

VOCAB_SIZE = 20
PAD_ID = 0
SEQ_LEN = 16
BATCH_SIZE = 8

# Fake data: random token sequences
train_data = torch.randint(1, VOCAB_SIZE, (100, SEQ_LEN))  # 100 "sentences"
print(f"Training data: {train_data.shape} (100 samples, {SEQ_LEN} tokens each)")

# 2. Create model
model = MiniGPT(VOCAB_SIZE, d_model=64, num_heads=4, num_layers=2)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# 3. Training loop
import torch.nn.functional as F

NUM_EPOCHS = 5
losses = []

model.train()
for epoch in range(NUM_EPOCHS):
    epoch_loss = 0.0
    num_batches = 0

    for i in range(0, len(train_data), BATCH_SIZE):
        batch = train_data[i:i+BATCH_SIZE]  # [batch_size, seq_len]

        # Prepare input and labels
        input_ids = batch[:, :-1]      # remove last token
        target_ids = batch[:, 1:]       # remove first token

        # Forward
        logits = model(input_ids)  # [batch, seq_len-1, vocab_size]

        # Loss: flatten all batch and position dimensions
        loss = F.cross_entropy(
            logits.reshape(-1, VOCAB_SIZE),  # [batch*(seq_len-1), vocab_size]
            target_ids.reshape(-1)            # [batch*(seq_len-1)]
        )

        # Backward
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()
        num_batches += 1

    avg_loss = epoch_loss / num_batches
    losses.append(avg_loss)
    print(f"Epoch {epoch+1}/{NUM_EPOCHS} | Loss: {avg_loss:.4f}")

print(f"\nLoss decreased from {losses[0]:.4f} to {losses[-1]:.4f} -> the model is learning!")

In [ ]:
# Visualize loss decrease
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 4))
plt.plot(losses, 'o-', markersize=8)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training loss curve')
plt.grid(True, alpha=0.3)
plt.show()

print("Loss is decreasing = the model is learning to predict the next token")

## 7. Core Review

```
+------------------------------------------------------------------+
|        What does "token-level" really mean in LLM training?      |
+------------------------------------------------------------------+
|                                                                    |
|  Input sequence: [BOS, I, love, you, China, EOS]                  |
|                    | remove last token                             |
|  Model input:   [BOS, I, love, you, China]                        |
|                    | model forward                                 |
|  Model output:  [logits0, logits1, ..., logits4]                   |
|                       each is a [vocab_size] score vector          |
|                    | compare against labels                        |
|  Expected:      [I,     love,   you,  China, EOS]                  |
|                    |      |       |      |      |                  |
|                    +------+-------+------+------+)                 |
|                    each position computes its own loss             |
|                    |                                                |
|  Loss = mean( loss0, loss1, loss2, ..., loss4 )                    |
|                    | backpropagation                                |
|  Gradients come from all 5 positions simultaneously                |
|  -> update model parameters                                        |
|                                                                    |
|  [x] Token-level: every token position contributes to loss         |
|  [x] Parallel: all positions predicted in one forward pass         |
|  [ ] NOT sentence-level: not just the last position               |
|  [ ] NOT sequential tokens: no waiting for previous tokens        |
|                                                                    |
+------------------------------------------------------------------+
```

## 8. Training vs Inference

| | During Training | During Inference/Generation |
|------|--------|------------|
| Input | Complete sentence (minus last token) | Only a prompt |
| Computation | All positions in parallel | Token by token, sequentially |
| Labels used | Ground truth (teacher forcing) | Previous self-generated token |
| Mask | Masks future tokens | Also masks future tokens |
| Loss | Computed at all token positions | No loss computation |

**Key differences**:
- Training uses teacher forcing -> all positions in parallel -> fast
- Inference has no ground truth -> must generate one token at a time -> slow (this is the fundamental reason LLM inference is slow)

-> Next Part: Inference / autoregressive generation!

## 9. The Gradient Perspective

So far we've been talking about loss, but loss is just a number. What actually drives model learning is the **gradient**.

```
loss (a single number)
    | backpropagation (backward)
gradient (one number per parameter)
    | optimizer (optimizer.step)
parameter update (model becomes slightly better)
```

This section dives into the gradient -- an often-skipped intermediate step -- to see what actually happens.

#### 9.1 Backpropagation: How Gradients Flow from Loss Back to Parameters

**Intuition**: loss is the "collective result" of all model parameters. Backpropagation asks:
> "If I increase this parameter by a tiny amount, how much does loss change?"

This "rate of change" is the gradient.

Let's understand this with the simplest possible example. Suppose we have a single neuron:

```
Input x --> [weight w] --> output y = w*x
                              |
                          loss = (y - target)^2
```

**Chain Rule**:
```
dloss/dw = dloss/dy * dy/dw
         = 2(y - target) * x
```

In an LLM, this chain passes through dozens of Transformer blocks, but the principle is exactly the same --
start from loss, trace back along the computation graph step by step, multiplying the local derivative at each operation.

In [ ]:
# Manual demonstration: backpropagation in a simple network
import torch

print("=== Manual Backpropagation Demo ===")
print()

# Build the simplest possible "model"
w = torch.tensor([0.5], requires_grad=True)
b = torch.tensor([0.1], requires_grad=True)

x = torch.tensor([2.0])   # input
target = torch.tensor([3.0])  # target

print(f"Parameters: w={w.item():.2f}, b={b.item():.2f}")
print(f"Input x={x.item():.2f}, Target={target.item():.2f}")
print()

# Forward
y = w * x + b              # y = 0.5*2 + 0.1 = 1.1
loss = (y - target) ** 2   # loss = (1.1 - 3)^2 = 3.61

print(f"Forward: y = w*x + b = {w.item()}*{x.item()} + {b.item()} = {y.item():.2f}")
print(f"Loss = (y - target)^2 = ({y.item():.2f} - {target.item():.2f})^2 = {loss.item():.2f}")
print()

# Backward
loss.backward()

print(f"dloss/dw = {w.grad.item():.4f}  (meaning: if w increases by 1, loss changes by {w.grad.item():.4f})")
print(f"dloss/db = {b.grad.item():.4f}  (meaning: if b increases by 1, loss changes by {b.grad.item():.4f})")
print()

# Manually verify the chain rule
print("=== Manual Verification of Chain Rule ===")
print(f"dloss/dy = 2*(y - target) = 2*({y.item():.2f} - {target.item():.2f}) = {2*(y.item()-target.item()):.2f}")
print(f"dy/dw = x = {x.item():.2f}")
print(f"dy/db = 1")
print(f"dloss/dw = dloss/dy * dy/dw = {2*(y.item()-target.item()):.2f} * {x.item():.2f} = {2*(y.item()-target.item())*x.item():.2f}")
print(f"PyTorch's dloss/dw = {w.grad.item():.4f}  Match!")

In [ ]:
# View gradient flow in MiniGPT
import torch
import torch.nn.functional as F

VOCAB_SIZE = 20
model = MiniGPT(VOCAB_SIZE, d_model=64, num_heads=4, num_layers=2)

dummy_input = torch.randint(1, VOCAB_SIZE, (2, 16))   # [batch=2, seq=16]
dummy_target = torch.randint(1, VOCAB_SIZE, (2, 15))  # [batch=2, seq=15]

# Forward
logits = model(dummy_input[:, :-1])
loss = F.cross_entropy(
    logits.reshape(-1, VOCAB_SIZE),
    dummy_target.reshape(-1)
)

# Backward
model.zero_grad()
loss.backward()

print("=== MiniGPT Layer Gradient Norms ===")
print()
print(f"{'Layer':<40s} {'Grad Norm':>12s} {'Param Shape':>18s}")
print("-" * 72)
total_grad_norm = 0
for name, param in model.named_parameters():
    if param.grad is not None:
        grad_norm = param.grad.norm().item()
        total_grad_norm += grad_norm ** 2
        param_shape = str(list(param.shape))
        print(f"{name:<40s} {grad_norm:>12.6f}  {param_shape:>18s}")

total_grad_norm = total_grad_norm ** 0.5
print("-" * 72)
print(f"{'Total gradient norm (L2)':<40s} {total_grad_norm:>12.6f}")
print()
print("Observations:")
print("  1. Every parameter has a gradient = backpropagation successfully delivered the loss signal to all layers")
print("  2. lm_head (output layer) gradients are typically larger = where the loss signal is strongest")
print("  3. Embedding layer gradients are smaller = signal attenuates through multiple layers")
print("  4. Residual connections ensure gradients don't vanish (this is key to training Transformers!)")

#### 9.2 Token-Level Gradients: Not All Tokens Are Equally Important

Earlier we said "every token contributes to loss." But wait: **do all tokens contribute equally?**

**No!** Consider this example:

```
Sentence: "The capital of France is Paris"
            |--easy--|  |med|  |key info|
            (low loss)         (high loss)
```

- Common words like "of", "is" are learned quickly -> low loss -> **small gradients**
- Key content words like "Paris" -> high loss -> **large gradients**

**This means the learning signal is primarily driven by "hard tokens," while easy tokens barely contribute gradients.**

This leads to the core problem found in RL training: **Advantage Collapsing** -- most rollouts have advantages near 0, resulting in weak gradient signals.

In [ ]:
# Demo: different token positions have different losses and gradients
import torch
import torch.nn.functional as F

print("=== Token-Level Loss Analysis ===")
print()

VOCAB_SIZE = 20
model = MiniGPT(VOCAB_SIZE, d_model=64, num_heads=4, num_layers=2)

# Construct a sentence: first half is an easy pattern, second half is random
# Easy pattern: [1,2,3,1,2,3,1,2,3] repeating
easy_part = torch.tensor([1, 2, 3, 1, 2, 3, 1, 2, 3])
# Random part
hard_part = torch.randint(10, VOCAB_SIZE, (7,))
sentence = torch.cat([easy_part, hard_part])  # seq_len=16
batch = sentence.unsqueeze(0)  # [1, 16]

input_ids = batch[:, :-1]      # [1, 15]
target_ids = batch[:, 1:]       # [1, 15]

print(f"First half (easy pattern): {easy_part.tolist()}")
print(f"Second half (random tokens): {hard_part.tolist()}")
print()

# Forward
logits = model(input_ids)  # [1, 15, vocab_size]

# Compute loss at each position
print("Loss at each token position:")
print(f"{'Position':>8s} {'Token':>6s} {'Loss':>10s} {'Region'}")
print("-" * 40)

for pos in range(15):
    pos_logits = logits[0, pos]  # [vocab_size]
    pos_target = target_ids[0, pos]
    pos_loss = F.cross_entropy(pos_logits.unsqueeze(0), pos_target.unsqueeze(0)).item()
    region = "Easy" if pos < 8 else "Hard"
    print(f"{pos:>8d} {pos_target.item():>6d} {pos_loss:>10.4f}  {region}")

# Compare average loss between easy and hard regions
logits_flat = logits.reshape(-1, VOCAB_SIZE)
targets_flat = target_ids.reshape(-1)
all_losses = F.cross_entropy(logits_flat, targets_flat, reduction='none')

easy_avg = all_losses[:9].mean().item()
hard_avg = all_losses[9:].mean().item()

print()
print(f"Easy region average loss: {easy_avg:.4f}")
print(f"Hard region average loss: {hard_avg:.4f}")
print(f"Hard region loss is {hard_avg/easy_avg:.2f}x the easy region")
print()
print("-> Hard tokens produce larger gradients, driving more learning.")
print("-> This is why RL training needs to focus on which rollouts/tokens truly contribute gradients.")

In [ ]:
# Advanced: visualizing token-level gradient distribution in LLMs
import torch

print("=== Token-Level Gradient Contribution Simulation ===")
print()

# Simulate gradient norms for each token in a sentence
torch.manual_seed(42)

# Simulate 20 tokens with increasing loss (low at start, high at end)
token_losses = torch.tensor([0.1, 0.15, 0.2, 0.15, 0.1,
                              0.3, 0.5, 0.8, 1.2, 1.5,
                              2.0, 2.5, 2.8, 3.0, 3.2,
                              3.5, 3.8, 4.0, 4.2, 4.5])

tokens = ["BOS", "I", "am", "an", "AI",
          "today", "weather", "is", "really", "nice",
          "quantum", "entangle-", "ment", "is", "non",
          "local", "physical", "phenom-", "enon", "EOS"]

# Gradient ~= loss (simplified: assume gradient is proportional to loss)
grad_contrib = token_losses / token_losses.sum() * 100

print("Token-level gradient contributions:")
print(f"{'Token':<12s} {'Loss':>8s} {'Grad%':>10s} {'Visualization'}")
print("-" * 60)

threshold = 5.0  # above 5% counts as "high contribution"
high_count = 0
for i in range(len(tokens)):
    bar_len = int(grad_contrib[i].item() * 3)
    bar = "#" * bar_len
    marker = " *" if grad_contrib[i] > threshold else ""
    if grad_contrib[i] > threshold:
        high_count += 1
    print(f"{tokens[i]:<12s} {token_losses[i].item():>8.2f} {grad_contrib[i].item():>8.1f}% {bar}{marker}")

print()
print(f"Tokens with gradient contribution > {threshold}%: {high_count}/{len(tokens)}")
print(f"These {high_count} tokens contribute {grad_contrib[-high_count:].sum().item():.1f}% of gradients")
print()
print("Key insights:")
print("  1. The first few tokens (common words) barely contribute gradients")
print("  2. The last few tokens (content words, hard words) contribute most gradients")
print("  3. Training efficiency is determined by the 'hardest few tokens'")
print("  4. Shuffle-R1's PTS + ABS specifically addresses this problem in RL")

#### 9.3 Gradient Clipping: Preventing Gradient Explosion

**Problem**: Sometimes a batch produces extremely large gradients (e.g., encountering a never-before-seen pattern). If the optimizer takes a full step with this gradient, parameters fly off -- loss becomes NaN, and training crashes.

**Solution**: Gradient clipping. Set an upper bound:

```
If gradient L2 norm > max_norm:
    Scale all gradients proportionally so norm = max_norm
Otherwise:
    Keep unchanged
```

```
       Gradient explosion
       ^
       |    /\             after clipping
       |   /  \           - - - -  max_norm
       |  /    \--       /
       | /             /
       |/-------     /
       +---------------> training step
       Without clipping, that spike would crash the model
```

Gradient clipping is almost always used in LLM training, typically with `max_norm=1.0`.

In [ ]:
# Demo: gradient clipping
import torch
import torch.nn as nn
import torch.nn.functional as F

print("=== Gradient Clipping Demo ===")
print()

# Create a small network, manually create an exploding gradient
linear = nn.Linear(10, 1)

x = torch.randn(5, 10)
target = torch.randn(5, 1) * 100  # deliberately scale up target to create large gradients

# Without clipping
loss = F.mse_loss(linear(x), target)
loss.backward()

raw_grad_norm = sum(p.grad.norm().item() ** 2 for p in linear.parameters()) ** 0.5
print(f"Gradient norm without clipping: {raw_grad_norm:.4f}")

# Reset
linear.zero_grad()

# With clipping
loss = F.mse_loss(linear(x), target)
loss.backward()
max_norm = 1.0
nn.utils.clip_grad_norm_(linear.parameters(), max_norm)
clipped_grad_norm = sum(p.grad.norm().item() ** 2 for p in linear.parameters()) ** 0.5

print(f"Gradient norm after clipping: {clipped_grad_norm:.4f} (limit={max_norm})")
print()
print(f"Original gradient too large -> clipped to {max_norm} -> training won't crash")
print()

# Practical: adding gradient clipping to the training loop
print("Practical code pattern:")
print("```python")
print("loss.backward()")
print("torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)")
print("optimizer.step()")
print("```")
print()
print("This clip line is your training 'seatbelt' --")
print("you won't notice it most of the time, but it saves your training at critical moments.")

#### 9.4 Gradient Accumulation: How Small GPUs Simulate Large Batches

**Problem**: LLM training needs large batches (e.g., 512), but your GPU can only fit batch=4. What do you do?

**Key observation**:
```
batch=8 gradient = batch=4 gradient + batch=4 gradient
                      ^ first batch       ^ second batch
```

Gradients can be "accumulated." So:

```
Target batch = 512, GPU can only run 32 at a time
-> Run 512/32 = 16 small batches, accumulating gradients without updating
-> After 16 runs, the gradient is equivalent to batch=512
-> Then call optimizer.step()
```

**Tradeoff**: Training becomes slower (16 forward passes per update), but at least it's possible.

**Difference from regular small batches**:
- Regular small batch: each forward -> backward -> step, the batch is genuinely small
- Gradient accumulation: multiple forward -> backward -> final step, **equivalent to a large batch**

In [ ]:
# Demo: gradient accumulation
import torch
import torch.nn.functional as F

print("=== Gradient Accumulation Demo ===")
print()

VOCAB_SIZE = 20
model_small = MiniGPT(VOCAB_SIZE, d_model=32, num_heads=2, num_layers=1)
model_large = MiniGPT(VOCAB_SIZE, d_model=32, num_heads=2, num_layers=1)

# Copy identical parameters
model_large.load_state_dict(model_small.state_dict())

# Fake data
all_data = torch.randint(1, VOCAB_SIZE, (16, 16))  # total 16 samples

# --- Method A: large batch (batch=16) directly ---
opt_large = torch.optim.SGD(model_large.parameters(), lr=0.01)

input_large = all_data[:, :-1]
target_large = all_data[:, 1:]
logits_large = model_large(input_large)
loss_large = F.cross_entropy(
    logits_large.reshape(-1, VOCAB_SIZE),
    target_large.reshape(-1)
)
opt_large.zero_grad()
loss_large.backward()

# Save large batch gradients
grads_large = {name: p.grad.clone() for name, p in model_large.named_parameters() if p.grad is not None}
opt_large.step()

# --- Method B: small batch (batch=4) + gradient accumulation 4 times ---
opt_small = torch.optim.SGD(model_small.parameters(), lr=0.01)
opt_small.zero_grad()

ACCUM_STEPS = 4
small_batch_size = 4
for step in range(ACCUM_STEPS):
    start = step * small_batch_size
    end = start + small_batch_size
    mini_batch = all_data[start:end]

    input_small = mini_batch[:, :-1]
    target_small = mini_batch[:, 1:]
    logits_small = model_small(input_small)
    loss_small = F.cross_entropy(
        logits_small.reshape(-1, VOCAB_SIZE),
        target_small.reshape(-1)
    )

    # Key: divide loss by accumulation steps to maintain consistent gradient scale
    (loss_small / ACCUM_STEPS).backward()
    print(f"  Accum step {step+1}/{ACCUM_STEPS}: loss={loss_small.item():.4f}, gradient accumulated (no update)")

print()

# Save accumulated small batch gradients
grads_small = {name: p.grad.clone() for name, p in model_small.named_parameters() if p.grad is not None}
opt_small.step()

# Compare gradients from both methods
print("=== Gradient Comparison ===")
all_close = True
for name in grads_large:
    diff = (grads_large[name] - grads_small[name]).norm().item()
    status = "Match" if diff < 1e-4 else "Diff"
    if diff >= 1e-4:
        all_close = False
    print(f"  {name:<30s} diff={diff:.8f} {status}")

print()
if all_close:
    print("Conclusion: accumulated gradients == large batch gradients")
    print("-> 4 forward passes with batch=4 simulated batch=16")
else:
    print("Note: minor differences may occur due to BatchNorm behavior in sequential mini-batches")
    print("But for LLMs (which only use LayerNorm), this is fully equivalent")

#### 9.5 The Complete Gradient Flow Picture in LLMs

Let's connect everything we've learned and trace the complete lifecycle of a gradient during LLM training:

```
+------------------------------------------------------------------+
|              Complete LLM Gradient Flow Diagram                    |
+------------------------------------------------------------------+
|                                                                    |
|  [Data]                                                            |
|    |                                                                |
|  [Forward: Input -> Embedding -> Transformer Blocks -> LM Head]    |
|    |                                                                |
|  [Loss: Cross-Entropy, one loss per token]                         |
|    |                                                                |
|    |  <- Key observation 1: Token-level gradients                  |
|    |     Easy tokens (of/is) -> low loss -> small gradients        |
|    |     Hard tokens (Paris) -> high loss -> large gradients       |
|    |     Training is mainly driven by "hard tokens"                |
|    |                                                                |
|  [Backward: gradients flow back from LM Head]                      |
|    |                                                                |
|    +-- LayerNorm: normalization, stable gradients                  |
|    +-- FFN: two Linear layers, gradients may be large              |
|    +-- Attention: QKV projections + Output projection              |
|    |     |                                                          |
|    |     +-- <- Key observation 2: attention head gradient diff.   |
|    |           Some heads have large gradients (active),            |
|    |           some have small gradients (redundant)               |
|    +-- Residual connection -> gradient shortcut, no vanishing      |
|    |                                                                |
|    | Back to Embedding layer (smallest gradients)                  |
|    |                                                                |
|  [Gradient Processing]                                             |
|    +-- Gradient Clipping: prevent explosion, max_norm~1.0          |
|    +-- Gradient Accumulation: small GPU simulates large batch      |
|    +-- (RL-specific) PTS+ABS: filter high-contribution rollouts    |
|    |                                                                |
|  [Parameter Update: optimizer.step()]                              |
|    |                                                                |
|  [Next batch, repeat]                                              |
|                                                                    |
+------------------------------------------------------------------+
```

**Three most important takeaways**:

1. **Residual connections = gradient highways**: Without them, gradients in deep Transformers would vanish (like in RNNs).
   Residual connections let gradients "skip" FFN and Attention, flowing directly backward.

2. **Token-level gradients are uneven**: Easy words barely contribute gradients; training is driven by hard words.
   This explains why LLMs improve slowly on "knowledge-intensive" tasks --
   knowledge words are a small fraction, so gradient signals are sparse.

3. **RL training amplifies gradient unevenness**: In SFT at least every token has a clear label,
   but in RL many rollouts have advantages near 0 -> even sparser gradient signals.
   Shuffle-R1's PTS+ABS was designed to address this.

In [ ]:
# Visualization: how residual connections protect gradient flow
print("=== Residual Connections and Gradient Flow ===")
print()

print("Without residual connections (like traditional networks):")
print("  Input -> Layer1 -> Layer2 -> ... -> Layer32 -> Output")
print("  Gradient: Output -> Layer32->...-> Layer1 -> Input")
print("  Problem: after 32 layers of multiplication, gradients may decay to 0 (vanishing gradients)")
print()

print("With residual connections (Transformer approach):")
print("  Input -> [Layer1 + Input] -> [Layer2 + prev] -> ... -> Output")
print("  Gradient: two paths --")
print("    Main path: Output -> Layer32 -> ... -> Layer1 -> Input  (may decay)")
print("    Shortcut: Output -> Input  (skips all layers!)  <- gradient highway")
print()
print("Because of the shortcut, at least some gradient reaches the bottom layers undamaged.")
print("This is why Transformers can be stacked to 100+ layers and still train.")
print()

# Simulate gradient decay at different depths
depths = [1, 4, 8, 16, 32, 64]

print("Simulation: initial gradient=1.0, decay after different numbers of layers")
print(f"{'Layers':>6s}  {'No residual':>12s}  {'With residual':>12s}")
print("-" * 32)

decay_per_layer = 0.95  # 5% decay per layer
skip_ratio = 0.3         # residual path accounts for 30%

for d in depths:
    no_skip = decay_per_layer ** d
    with_skip = no_skip * (1 - skip_ratio) + skip_ratio
    print(f"{d:>6d}  {no_skip:>12.6f}  {with_skip:>12.6f}")

print()
print("Conclusion: after 32 layers, gradient without residuals is only 20%, with residuals it's still 44%")
print("The more layers, the more valuable residual connections become.")

#### 9.6 Gradient Perspective Summary

| Concept | One-liner | Why it matters |
|:---|:---|:---|
| **Backpropagation** | Loss -> chain rule -> gradient per parameter | Foundation of training; no learning without it |
| **Token-level gradients** | Hard tokens get large gradients, easy tokens near 0 | Explains why training efficiency is driven by "hard samples" |
| **Gradient clipping** | Scale down proportionally when norm > max_norm | Safety belt against training crashes |
| **Gradient accumulation** | Accumulate gradients from multiple small batches, update once | Lifesaving technique for training large models on small GPUs |
| **Residual connections** | Gradients can skip layers and flow directly backward | Fundamental reason Transformers can be 100+ layers deep |
| **RL gradient sparsity** | Even fewer effective gradients in RL training | Motivation for work like Shuffle-R1 |

**Gradients are the "translator" between loss and parameter updates** --
they translate "where this batch performed poorly" into "how each parameter should be adjusted."
Understanding gradients is essential to truly understanding how models learn.

## 10. From Conversations to Tokens

> **Analogy**: You want to write a letter to a friend. You have "what you want to say" in your head, but the post office only accepts "words written on paper." Chat Template is that "format for turning thoughts into a letter" -- you write "Dear So-and-so" first, then the body, then sign off.
> LLMs work the same way: your conversational messages can't be fed directly to the model; they must first be "formatted" into tokens via Chat Template.

#### 10.1 What Real Training Data Looks Like

Real LLM training data comes in JSONL format (one JSON object per line), which is the OpenAI API-compatible standard:

```jsonl
{"messages": [{"role": "system", "content": "You are a math assistant"}, {"role": "user", "content": "1+1=?"}, {"role": "assistant", "content": "1+1=2"}]}
{"messages": [{"role": "user", "content": "Hello"}, {"role": "assistant", "content": "Hello! How can I help?"}]}
{"messages": [{"role": "system", "content": "You are a translator"}, {"role": "user", "content": "Hello"}, {"role": "assistant", "content": "Ni hao"}]}
```

**Core question**: how does the JSON object above become the `input_ids` and `labels` that the model sees?

We need to understand three things:
1. **Concatenation**: How is the messages list concatenated into a continuous token sequence?
2. **Segmentation**: Which tokens are "context for the model to read" vs. "answers for the model to learn"?
3. **Loss**: How are special markers (like `<|im_start|>`) handled during loss computation?

Let's walk through this step by step using the `transformers` library.

In [ ]:
# ============================================================
# Using the real transformers library to see what Chat Template does
# ============================================================
print("=== Real tokenizer demo: How Chat Template turns messages into tokens ===\n")

# Try loading Qwen2.5's tokenizer (auto-downloads on first run)
try:
    from transformers import AutoTokenizer
    
    # Qwen2.5-0.5B's tokenizer is small (~30MB), downloads quickly
    # Its chat template uses ChatML style, consistent with DeepSeek and Qwen series
    MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
    
    print(f"Loading tokenizer: {MODEL_NAME}")
    print("(First run will download ~30MB, please wait...)\n")
    
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
    
    # First, see what special tokens the tokenizer has
    print("=== Tokenizer Special Tokens ===")
    print(f"  bos_token:        {repr(tokenizer.bos_token)} -> id={tokenizer.bos_token_id}")
    print(f"  eos_token:        {repr(tokenizer.eos_token)} -> id={tokenizer.eos_token_id}")
    print(f"  pad_token:        {repr(tokenizer.pad_token)} -> id={tokenizer.pad_token_id}")
    print(f"  Vocabulary size:  {len(tokenizer)} tokens")
    print()
    
    # Look at the chat template itself (it's just a Jinja2 template string!)
    print("=== Chat Template (Jinja2 template) ===")
    ct = tokenizer.chat_template
    if ct:
        # Only show first 500 characters
        print(ct[:500])
        print("...")
    print()
    
    # ============================================================
    # Core demo: what does apply_chat_template actually do?
    # ============================================================
    messages = [
        {"role": "system", "content": "You are a math assistant"},
        {"role": "user", "content": "1+1=?"},
        {"role": "assistant", "content": "1+1=2"},
    ]
    
    print("=== Input: messages list ===")
    import json
    print(json.dumps(messages, ensure_ascii=False, indent=2))
    print()
    
    # Step 1: tokenize=False -- see the rendered text (human-readable)
    print("=== Step 1: apply_chat_template(tokenize=False) -- render to text ===")
    rendered_text = tokenizer.apply_chat_template(
        messages, 
        tokenize=False,            # don't tokenize, just show text
        add_generation_prompt=False # no generation prompt (not needed for training)
    )
    print("Rendered text (note the special token positions):")
    print(repr(rendered_text))
    print()
    print("Visualized:")
    print(rendered_text)
    print()
    
    # Step 2: tokenize=True -- directly get input_ids
    print("=== Step 2: apply_chat_template(tokenize=True) -- directly get token IDs ===")
    input_ids = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=False,
        return_tensors="pt"  # return PyTorch tensor
    )
    print(f"input_ids shape: {input_ids.shape}")  # [batch=1, seq_len]
    print(f"input_ids content: {input_ids[0].tolist()}")
    print(f"Sequence length: {len(input_ids[0])}")
    print()
    
    # Step 3: decode each token to see what each segment is
    print("=== Step 3: Decode each token -- see what each segment is ===")
    print(f"{'Position':<8s} {'Token ID':>8s} {'Decoded text':<30s} {'Description'}")
    print("-" * 75)
    
    for i, tid in enumerate(input_ids[0].tolist()):
        decoded = tokenizer.decode([tid])
        # Identify token type
        if tid == tokenizer.bos_token_id:
            note = "<- BOS (start token)"
        elif tid == tokenizer.eos_token_id:
            note = "<- EOS (end token)"
        elif tid == 151644:  # Qwen <|im_start|>
            note = "<- <|im_start|> special token"
        elif tid == 151645:  # Qwen <|im_end|>
            note = "<- <|im_end|> special token"
        elif tid >= len(tokenizer) - 20:  # special tokens are often near the end
            note = "<- special token"
        else:
            note = ""
        print(f"{i:<8d} {tid:>8d} {decoded:<30s} {note}")

    print(f"\nKey observations:")
    print(f"  1. The text becomes one continuous token sequence, not three separate arrays")
    print(f"  2. system/user/assistant turns are separated by <|im_start|> and <|im_end|>")
    print(f"  3. The sequence contains the system prompt, user question, and assistant answer")
    print(f"  4. The model sees the whole sequence at once through a causal attention mask")

except ImportError:
    print("transformers is not installed. Run: pip install transformers")
    print()
    print("Manually simulating Qwen2.5 chat template behavior...")
    print()
    
    print("Qwen2.5 Chat Template rendering rules (ChatML format):")
    print('  <|im_start|>system')
    print('  {system_content}')
    print('  <|im_end|>')
    print('  <|im_start|>user')
    print('  {user_content}')
    print('  <|im_end|>')
    print('  <|im_start|>assistant')
    print('  {assistant_content}')
    print('  <|im_end|>')
    print()
    
    print("Use the simplified implementation above to inspect how role markers and text become one sequence.")

except Exception as e:
    print(f"Loading failed: {e}")
    print("Using the simplified demo instead...")


#### 10.2 Controlled Experiment: Manual Concatenation vs Official API

Above we used `tokenizer.apply_chat_template()` to do everything in one step. But what does it actually do internally? Are there any hidden operations?

Let's do a **controlled experiment**:

| Method | How it works |
|------|--------|
| Method 1 (Official) | Call `tokenizer.apply_chat_template(messages)` |
| Method 2 (Manual) | Manually concatenate strings in ChatML format, then `tokenizer.encode()` |

**If both produce the same result, it proves that `apply_chat_template` has no magic internally -- it's just string concatenation + tokenization.**

Using the same example:
```json
{"messages": [
    {"role": "system", "content": "You are a math assistant"},
    {"role": "user", "content": "1+1=?"},
    {"role": "assistant", "content": "1+1=2"}
]}
```

ChatML format template rules (this is what the Jinja2 template does):
```
Each message -> <|im_start|>{role}\n{content}<|im_end|>\n
```

Broken into steps:

```
Step 1: system message
  <|im_start|>system\nYou are a math assistant<|im_end|>\n
  \---+---/  \-+-/  \--+-/  \------+---/  \--+--/ \+/
  special    role   newline  content   special  newline

Step 2: user message (immediately after system)
  ...<|im_end|>\n<|im_start|>user\n1+1=?<|im_end|>\n
     \--system message end--/  \--user message start---/

Step 3: assistant message (immediately after user)
  ...<|im_end|>\n<|im_start|>assistant\n1+1=2<|im_end|>\n
     \--user message end----/  \----assistant message-----/
```

**Key point**: All messages are concatenated into **a single continuous text**, with no spaces, line break separators, or array markers.
The model uses special tokens like `<|im_start|>` and `<|im_end|>` to identify message boundaries.

Below we'll use both methods side by side, printing intermediate steps, then comparing item by item:

In [ ]:
# ============================================================
# 10.3 Controlled experiment: manual concatenation vs official API -- proving they're identical
# ============================================================
# Core idea: apply_chat_template has no magic internally, it's just string concatenation in ChatML format.
# We manually concatenate once, then compare with the official result item by item.

# If the real transformers tokenizer wasn't loaded, use a simplified offline version.
if "tokenizer" not in globals():
    print("Real tokenizer not detected, using SimpleChatTokenizer for offline demo.")

    class SimpleChatTokenizer:
        """Minimal functional ChatML tokenizer to prove template = string concatenation + tokenize"""
        def __init__(self):
            self.special = {"<|im_start|>": 100001, "<|im_end|>": 100002}
            self.vocab = {"\n": 10}
            self.reverse = {10: "\n", 100001: "<|im_start|>", 100002: "<|im_end|>"}

        def convert_tokens_to_ids(self, token):
            return self.special.get(token, self.vocab.get(token, -1))

        def apply_chat_template(self, messages, tokenize=False):
            text = "".join(
                f"<|im_start|>{m['role']}\n{m['content']}<|im_end|>\n"
                for m in messages
            )
            return self.encode(text, add_special_tokens=False) if tokenize else text

        def encode(self, text, add_special_tokens=False):
            ids = []
            i = 0
            while i < len(text):
                matched = False
                for tok, tid in self.special.items():
                    if text.startswith(tok, i):
                        ids.append(tid)
                        i += len(tok)
                        matched = True
                        break
                if matched:
                    continue
                ch = text[i]
                if ch not in self.vocab:
                    self.vocab[ch] = 1000 + len(self.vocab)
                    self.reverse[self.vocab[ch]] = ch
                ids.append(self.vocab[ch])
                i += 1
            return ids

        def decode(self, ids):
            return "".join(self.reverse[i] for i in ids)

    tokenizer = SimpleChatTokenizer()

print("=" * 70)
print("Controlled experiment: manual concatenation vs official apply_chat_template")
print("=" * 70)

# ============================================================
# Prepare conversation data
# ============================================================
messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "What is 1+1?"},
    {"role": "assistant", "content": "1+1=2."},
]

print("\nOriginal conversation:")
for i, msg in enumerate(messages):
    print(f"  [{i}] {msg['role']}: {msg['content']}")

# ============================================================
# First, look at special token IDs (the skeleton of ChatML format)
# ============================================================
IM_START_ID = tokenizer.convert_tokens_to_ids("<|im_start|>")
IM_END_ID   = tokenizer.convert_tokens_to_ids("<|im_end|>")
NEWLINE_ID  = tokenizer.convert_tokens_to_ids("\n")

print(f"\nSpecial token IDs:")
print(f"  '<|im_start|>' -> ID = {IM_START_ID}")
print(f"  '<|im_end|>'   -> ID = {IM_END_ID}")
print(f"  '\\n'           -> ID = {NEWLINE_ID}")

#### 10.3 The Crucial Step: Constructing Labels -- How Special Tokens Are Handled in Loss

Now we have `input_ids` (all tokens the model sees), but training also needs `labels` (telling the model "which tokens should you learn").

**Analogy**: An English exam gives you a reading passage + questions + reference answers.
- The passage (system prompt) -> you read it, but don't need to memorize it -> labels = IGNORE
- The questions (user message) -> you read them, but don't need to recite them -> labels = IGNORE
- The answers (assistant message) -> this is what you need to learn to write -> labels = real token IDs
- Punctuation/formatting (special tokens) -> these are formatting symbols, don't need to predict -> labels = IGNORE

```
input_ids: [151644, 8948, 198, 9942, 10603, 107659, 113738, 151645, 198, 
            151644, 872, 198, 16, 17, 18, 19, 20, 151645, 198,
            151644, 78191, 198, 16, 17, 18, 19, 18, 151645, 198]
           |-- system frame+content --||-- user frame+content --||- assistant content -|

labels:    [-100, -100, -100, -100, -100, -100, -100, -100, -100,
            -100, -100, -100, -100, -100, -100, -100, -100, -100, -100,
            -100, -100, -100, 16,   17,   18,   19,   18,  -100, -100]
           ^ All ignored in loss (system+user+special tokens)    ^ Only here   ^ Also ignored
```

**Why use -100 for labels?**
PyTorch's `CrossEntropyLoss` has an `ignore_index` parameter, defaulting to `-100`.
All positions where the label equals `ignore_index`: no loss is computed, no gradient is produced.

```
CrossEntropyLoss(ignore_index=-100) behavior:
  label = 5    -> normal computation: loss = -log(pred[5])
  label = -100 -> skip: loss = 0, grad = 0
```

Below we construct labels in code and verify that loss computation indeed skips the -100 positions:

In [ ]:
# ============================================================
# Label construction + ignore_index internals
# ============================================================
import torch
import torch.nn.functional as F

print("=== Label construction + ignore_index verification ===\n")

# Reuse previous vocab and tokenization
vocab = {
    "<|im_start|>": 151644, "<|im_end|>": 151645,
    "system": 8948, "user": 872, "assistant": 78191,
    "You": 9942, "are": 10603, "math": 107659, "assistant_w": 113738,
    "1": 16, "+": 17, "2": 18, "=": 19, "?": 20, "\n": 198,
}
id_to_word = {v: k for k, v in vocab.items()}

def encode(text):
    tokens = []
    i = 0
    while i < len(text):
        matched = None
        for word in sorted(vocab.keys(), key=lambda x: -len(x)):
            if text[i:].startswith(word):
                matched = word
                break
        if matched:
            tokens.append(vocab[matched])
            i += len(matched)
        else:
            tokens.append(0)
            i += 1
    return tokens

messages = [
    {"role": "system", "content": "You are math assistant"},
    {"role": "user", "content": "1+1=?"},
    {"role": "assistant", "content": "1+1=2"},
]

IM_START = "<|im_start|>"
IM_END = "<|im_end|>"
IGNORE = -100  # PyTorch's default ignore_index

# ============================================================
# Step 1: Construct input_ids
# ============================================================
print("Step 1: Construct input_ids")
all_text = ""
for msg in messages:
    all_text += f"{IM_START}{msg['role']}\n{msg['content']}{IM_END}\n"

input_ids = torch.tensor([encode(all_text)])
print(f"  input_ids: {input_ids[0].tolist()}")
print()

# ============================================================
# Step 2: Construct labels -- track each token's source
# ============================================================
print("Step 2: Construct labels (annotate each token)")
print()

labels = torch.full_like(input_ids, IGNORE)  # start with all IGNORE

pos = 0
for msg in messages:
    role = msg["role"]
    content = msg["content"]

    # Header: <|im_start|>role\n -> IGNORE (already IGNORE by default)
    header = f"{IM_START}{role}\n"
    hlen = len(encode(header))
    pos += hlen

    # Content
    cids = encode(content)
    if role == "assistant":
        # Only assistant content gets real labels
        for cid in cids:
            labels[0, pos] = cid
            pos += 1
    else:
        # system/user content stays IGNORE
        pos += len(cids)

    # Footer: <|im_end|>\n -> IGNORE
    footer = f"{IM_END}\n"
    flen = len(encode(footer))
    pos += flen

print(f"  input_ids: {input_ids[0].tolist()}")
print(f"  labels:    {labels[0].tolist()}")
print()

# Token-by-token comparison
print("Token-by-token comparison (L=loss, .=ignore):")
print(f"  {'Pos':<4s} {'input_id':>8s} {'token':<18s} {'label':>8s} {'Source'}")
print(f"  {'----'} {'--------'} {'------------------'} {'--------'} {'-'*30}")

pos = 0
for msg in messages:
    role = msg["role"]
    content = msg["content"]

    header_ids = encode(f"{IM_START}{role}\n")
    for hid in header_ids:
        word = id_to_word.get(hid, "???")
        lid = labels[0, pos].item()
        status = "IGNORE" if lid == IGNORE else f"{lid}"
        print(f"  {pos:<4d} {hid:>8d} {word:<18s} {status:>8s} header({role})")
        pos += 1

    content_ids = encode(content)
    for cid in content_ids:
        word = id_to_word.get(cid, "???")
        lid = labels[0, pos].item()
        status = "IGNORE" if lid == IGNORE else f"{lid}"
        marker = " <-- LEARN" if lid != IGNORE else ""
        print(f"  {pos:<4d} {cid:>8d} {word:<18s} {status:>8s} content({role}){marker}")
        pos += 1

    footer_ids = encode(f"{IM_END}\n")
    for fid in footer_ids:
        word = id_to_word.get(fid, "???")
        lid = labels[0, pos].item()
        status = "IGNORE" if lid == IGNORE else f"{lid}"
        print(f"  {pos:<4d} {fid:>8d} {word:<18s} {status:>8s} footer")
        pos += 1

# Verify: only assistant content has real labels
n_learn = sum(1 for l in labels[0].tolist() if l != IGNORE)
n_total = len(labels[0])
print(f"\nTotal tokens: {n_total}, tokens contributing to loss: {n_learn} ({n_learn/n_total*100:.0f}%)")
print("Only the assistant's content tokens contribute to loss")

#### 10.4 Multi-Turn Conversations: How Long Conversations Are Concatenated

Real training data often contains multi-turn conversations. For example:
```json
{"messages": [
    {"role": "system", "content": "You are a math teacher"},
    {"role": "user", "content": "1+1=?"},
    {"role": "assistant", "content": "1+1=2"},
    {"role": "user", "content": "What about 2+2?"},
    {"role": "assistant", "content": "2+2=4"}
]}
```

**The concatenation rule is exactly the same**: all messages are concatenated in order into a single continuous token sequence.

```
<|im_start|>system\nYou are a math teacher<|im_end|>\n
<|im_start|>user\n1+1=?<|im_end|>\n
<|im_start|>assistant\n1+1=2<|im_end|>\n
<|im_start|>user\nWhat about 2+2?<|im_end|>\n
<|im_start|>assistant\n2+2=4<|im_end|>\n
```

**Labels are also the same**: each turn's assistant content participates in loss computation; system/user/special markers are all ignored.

```
+------------------------------------------------------------------+
|              Multi-turn conversation labels rule                    |
+------------------------------------------------------------------+
|                                                                    |
|  Turn 1: user->"1+1=?"        <- model reads, doesn't learn       |
|           assistant->"1+1=2"   <- model reads, must learn!        |
|                                                                    |
|  Turn 2: user->"What about 2+2?" <- model reads, doesn't learn    |
|           assistant->"2+2=4"   <- model reads, must learn!        |
|                                                                    |
|  Through attention, the model can see the full history from Turn 1 |
|  -> the model learns to "answer based on conversation history"     |
|                                                                    |
+------------------------------------------------------------------+
```

**Why is multi-turn data important?**
- Single-turn "one question, one answer" -> model can only give one response
- Multi-turn "continuous conversation" -> model learns: follow-up questions, clarification, remembering context
- Real training typically mixes: ~60% multi-turn + ~40% single-turn

Below is a demo of multi-turn conversation concatenation and label construction:

In [ ]:
# ============================================================
# Multi-turn concatenation proof: 5 messages also become one segment
# ============================================================

print("=" * 70)
print("Proof: multi-turn conversations are also concatenated into one continuous token sequence")
print("=" * 70)
print()

# Vocabulary
vocab = {
    "<|im_start|>": 151644, "<|im_end|>": 151645,
    "system": 8948, "user": 872, "assistant": 78191,
    "You": 9942, "are": 10603, "math": 107659, "teacher": 113740,
    "1": 16, "+": 17, "2": 18, "=": 19, "?": 20, ".": 21,
    "3": 22, "4": 23, "What": 104322, "about": 104535, "\n": 198,
}
id_to_word = {v: k for k, v in vocab.items()}

def encode(text):
    tokens = []
    i = 0
    while i < len(text):
        matched = None
        for word in sorted(vocab.keys(), key=lambda x: -len(x)):
            if text[i:].startswith(word):
                matched = word
                break
        if matched:
            tokens.append(vocab[matched])
            i += len(matched)
        else:
            tokens.append(0)
            i += 1
    return tokens

IM_START = "<|im_start|>"
IM_END = "<|im_end|>"
IGNORE = -100

# Multi-turn conversation data (2 turns)
multi_turn = {
    "messages": [
        {"role": "system", "content": "You are a math teacher"},
        {"role": "user", "content": "1+1=?"},
        {"role": "assistant", "content": "1+1=2."},
        {"role": "user", "content": "What about 2+2?"},
        {"role": "assistant", "content": "2+2=4."},
    ]
}

print("Multi-turn conversation data (5 messages = system + 2 turns x 2):")
for i, msg in enumerate(multi_turn["messages"]):
    print(f"  [{i}] {msg['role']:>10s}: {msg['content']}")

# ============================================================
# Step-by-step concatenation -- 5 messages
# ============================================================
print()
print("=" * 70)
print("Step-by-step concatenation -- all 5 messages into one segment")
print("=" * 70)

all_text = ""
all_ids = []

for step, msg in enumerate(multi_turn["messages"]):
    role = msg["role"]
    content = msg["content"]

    segment = f"{IM_START}{role}\n{content}{IM_END}\n"
    segment_ids = encode(segment)

    before_len = len(all_ids)
    all_text += segment
    all_ids.extend(segment_ids)

    print(f"\nStep {step+1}/5: concatenate {role} -> \"{content}\"")
    print(f"  Segment: {repr(segment)}")
    print(f"  Segment token count: {len(segment_ids)}")
    print(f"  Cumulative token count: {before_len} -> {len(all_ids)}")
    print(f"  Cumulative text: {repr(all_text)}")

# ============================================================
# Final proof
# ============================================================
print(f"\n{'='*70}")
print("Result: 5 messages concatenated into 1 continuous token sequence")
print(f"{'='*70}")
print(f"  Total tokens: {len(all_ids)}")
print(f"  Complete IDs: {all_ids}")
print()

# Token-by-token turn annotation
print("Token-by-token annotation (proving it's a continuous sequence, not segmented arrays):")
print(f"{'Pos':<4s} {'ID':>7s} {'token':<16s} {'Turn/Role':<25s} {'Loss?'}")
print(f"{'----'} {'-------'} {'----------------'} {'-'*25} {'-'*8}")

pos = 0
for turn_idx, msg in enumerate(multi_turn["messages"]):
    role = msg["role"]
    content = msg["content"]

    header_ids = encode(f"{IM_START}{role}\n")
    for hid in header_ids:
        word = id_to_word.get(hid, "???")
        print(f"{pos:<4d} {hid:>7d} {word:<16s} {role:<25s} ignore")
        pos += 1

    content_ids = encode(content)
    learn = "LEARN" if role == "assistant" else "ignore"
    for cid in content_ids:
        word = id_to_word.get(cid, "???")
        print(f"{pos:<4d} {cid:>7d} {word:<16s} {role:<25s} {learn}")
        pos += 1

    footer_ids = encode(f"{IM_END}\n")
    for fid in footer_ids:
        word = id_to_word.get(fid, "???")
        print(f"{pos:<4d} {fid:>7d} {word:<16s} {'footer':<25s} ignore")
        pos += 1

print(f"\nTotal: {pos} tokens, all from one continuous sequence")

#### 10.5 The Complete Training Loop: Connecting Chat Template with Training

Recall the training loop from earlier: `input_ids = batch[:, :-1]`, `labels = batch[:, 1:]`.

That was simplified training with "raw text." For **conversational data**, the complete flow is:

```
+------------------------------------------------------------------+
|          Chat Template Training Loop (Complete Version)             |
+------------------------------------------------------------------+
|                                                                    |
|  1. Read JSONL data                                                |
|     {"messages": [{role, content}, ...]}                           |
|                                                                    |
|  2. apply_chat_template(messages)                                  |
|     -> input_ids:  [151644, 8948, ..., 151645, 198, ...]          |
|     -> labels:     [-100,   -100, ...,  16,    17,   18, ...]      |
|                                                                    |
|  3. Data preparation (same as beginning of Part 5)                 |
|     input_ids = tensor[:, :-1]   # remove last token              |
|     labels = labels[:, 1:]        # shift right by one             |
|                                                                    |
|  4. Forward + Loss                                                 |
|     logits = model(input_ids)                                      |
|     loss = CrossEntropyLoss(logits, labels, ignore_index=-100)     |
|                                                                    |
|  5. Backward + Update                                              |
|     loss.backward()                                                |
|     clip_grad_norm_(...)                                           |
|     optimizer.step()                                               |
|                                                                    |
+------------------------------------------------------------------+
```

#### 10.6 How Does This Connect to Part 3 (Generation)?

After training, inference uses the **same chat template**:

```python
# During inference (autoregressive generation from 07-generation notebook)
messages = [
    {"role": "system", "content": "You are a math assistant"},
    {"role": "user", "content": "3+3=?"}
]

# Same apply_chat_template (but with add_generation_prompt=True)
# This adds <|im_start|>assistant\n at the end -- signaling "your turn to speak"
prompt_ids = tokenizer.apply_chat_template(
    messages, tokenize=True, add_generation_prompt=True
)

# Then comes the autoregressive generation from Part 3:
# model.generate(prompt_ids, temperature=0.7, top_p=0.9, ...)
```

**The complete loop**:
> During training, chat template concatenates conversations into token sequences -> model learns assistant response patterns
> -> During inference, chat template formats user input into a prompt -> model autoregressively generates assistant responses
> -> Frontend strips special tokens -> returns plain text to the user

Below is a complete training loop to wrap up:

In [ ]:
# ============================================================
# Complete training loop: Chat Template + MiniGPT
# ============================================================
import torch
import torch.nn as nn
import torch.nn.functional as F

print("=== Complete training loop: conversation data -> tokens -> training ===\n")

# -------- Reuse previous vocab and functions --------
vocab = {
    "<|im_start|>": 151644, "<|im_end|>": 151645,
    "system": 8948, "user": 872, "assistant": 78191,
    "You": 9942, "are": 10603, "math": 107659, "teacher": 113740,
    "1": 16, "+": 17, "2": 18, "=": 19, "?": 20, ".": 21,
    "3": 22, "4": 23, "What": 104322, "about": 104535, "\n": 198,
    "translate": 112345, "or": 105678,
    "Hello": 201, "Ni_hao": 202, "!": 203,
    "weather": 301, "how": 302, "today": 303,
    "nice": 304, "sunny": 305,
}
id_to_word = {v: k for k, v in vocab.items()}
VOCAB_SIZE = max(vocab.values()) + 10
IM_START = "<|im_start|>"
IM_END = "<|im_end|>"
IGNORE = -100
PAD_ID = 0

def encode(text):
    tokens = []
    i = 0
    while i < len(text):
        matched = None
        for word in sorted(vocab.keys(), key=lambda x: -len(x)):
            if text[i:].startswith(word):
                matched = word
                break
        if matched:
            tokens.append(vocab[matched])
            i += len(matched)
        else:
            tokens.append(0)
            i += 1
    return tokens

# -------- Prepare training data (3 conversations) --------
train_conversations = [
    {
        "messages": [
            {"role": "system", "content": "You are a math teacher"},
            {"role": "user", "content": "1+1=?"},
            {"role": "assistant", "content": "1+1=2."},
        ]
    },
    {
        "messages": [
            {"role": "system", "content": "You are translate or"},
            {"role": "user", "content": "Hello!"},
            {"role": "assistant", "content": "Ni_hao!"},
        ]
    },
    {
        "messages": [
            {"role": "user", "content": "weather how today?"},
            {"role": "assistant", "content": "today nice sunny."},
        ]
    },
]

# -------- Construct input_ids and labels --------
print("=== Constructing training data ===")
all_inputs = []
all_labels = []

for idx, conv in enumerate(train_conversations):
    messages = conv["messages"]

    # Concatenate text
    text = ""
    for msg in messages:
        text += f"{IM_START}{msg['role']}\n{msg['content']}{IM_END}\n"

    input_ids = encode(text)
    labels = [IGNORE] * len(input_ids)

    # Annotate assistant content
    pos = 0
    for msg in messages:
        role = msg["role"]
        content = msg["content"]
        pos += len(encode(f"{IM_START}{role}\n"))
        cids = encode(content)
        if role == "assistant":
            for j, cid in enumerate(cids):
                labels[pos + j] = cid
        pos += len(cids)
        pos += len(encode(f"{IM_END}\n"))

    all_inputs.append(input_ids)
    all_labels.append(labels)

    n_assistant = sum(1 for l in labels if l != IGNORE)
    n_total = len(labels)
    print(f"Conversation {idx+1}: {n_total} tokens, {n_assistant} contribute to loss ({n_assistant/n_total*100:.0f}%)")

# -------- Pad to same length and create batch --------
max_len = max(len(ids) for ids in all_inputs)
batch_input = torch.full((len(all_inputs), max_len - 1), PAD_ID)
batch_labels = torch.full((len(all_inputs), max_len - 1), IGNORE)

for i in range(len(all_inputs)):
    inp = torch.tensor(all_inputs[i][:-1])  # remove last token
    lab = torch.tensor(all_labels[i][1:])    # shift right
    batch_input[i, :len(inp)] = inp
    batch_labels[i, :len(lab)] = lab

print(f"\nBatch input shape: {batch_input.shape}")
print(f"Batch labels shape: {batch_labels.shape}")

# -------- Train --------
model = MiniGPT(VOCAB_SIZE, d_model=64, num_heads=4, num_layers=2)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

print(f"\nModel parameters: {sum(p.numel() for p in model.parameters()):,}")
print("\nTraining for 20 epochs...")

model.train()
for epoch in range(20):
    logits = model(batch_input)
    loss = F.cross_entropy(
        logits.reshape(-1, VOCAB_SIZE),
        batch_labels.reshape(-1),
        ignore_index=IGNORE
    )
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if (epoch + 1) % 5 == 0:
        print(f"  Epoch {epoch+1}/20 | Loss: {loss.item():.4f}")

print("\nTraining complete!")
print("Key points:")
print("  1. Only assistant content tokens contribute to loss (ignore_index=-100)")
print("  2. All messages are concatenated into one continuous sequence")
print("  3. System/user content provides context but no learning signal")
print("  4. This is how real chat models are trained (ChatML format)")

## 11. Training Stability

So far we've covered how loss is computed and how gradients flow. But in actual training, there are three engineering techniques that are almost always needed. They don't solve the problem of "what should the model learn," but rather "will the training process itself crash."

There are three main scenarios for training instability:

- **A gradient spike at some step**: e.g., a batch happens to contain extreme data, or gradients in deep layers amplify during backpropagation. Without intervention, a single parameter update can throw the model from a normal state to a completely random one, and loss becomes NaN.
- **Not enough memory but need large batches**: small batch sizes make gradient directions unstable, each step moving toward a direction that only represents a few samples; but increasing batch size requires exponentially more memory, and a single GPU can't fit it.
- **Random initial direction at the start of training**: initial parameters are random, so gradient directions are random too. If the first step uses the full learning rate, it's like sprinting blindfolded -- easy to fall into a steep valley on the loss surface and never climb out.

Three techniques address these three problems:

| Technique | Problem Solved | Core Mechanism |
|:---|:---|:---|
| Gradient Clipping | Gradient explosion, oversized single-step updates | When total gradient norm exceeds threshold, scale proportionally; direction unchanged, step size limited |
| Gradient Accumulation | Insufficient memory, small batches give inaccurate gradients | Accumulate gradients from multiple small batches, update once; equivalent to large batch |
| Warmup | Random initial direction, large LR easily goes off course | First N steps linearly increase LR from 0; take small steps to explore direction first, then accelerate |

The three techniques solve different problems but share the same goal: **keep every step of training controllable, preventing deviation due to unexpected large gradients, small batches, or incorrect initial direction.**

#### 11.1 Gradient Clipping: Installing a "Speed Limiter" for Gradients

During training, certain data (e.g., a sudden garbled passage in an article) can cause a batch to produce very large gradients.

If we update parameters with this huge gradient directly, the model can "fly off" in one step -- loss suddenly spikes and never recovers.

**Gradient Clipping approach**: after computing gradients, check their total magnitude (norm). If it exceeds a threshold (e.g., 1.0), scale them down proportionally so the total magnitude equals the threshold.

```
Original gradient: [3.0, -5.0, 2.0, ...]  -> norm = 6.2
Threshold clip = 1.0
Scale factor = 1.0 / 6.2 = 0.161
After clipping:  [0.48, -0.81, 0.32, ...]  -> norm = 1.0
```

Direction unchanged, just smaller steps. Like a car's speed limiter -- it doesn't change direction, only limits maximum speed.

In [ ]:
# === Gradient Clipping hand calculation + code demo ===
import torch
import torch.nn as nn

print("=== Gradient Clipping Hand Calculation ===")
print()

# Simulate a gradient vector
grad = torch.tensor([3.0, -5.0, 2.0, -1.0, 4.0])
max_norm = 1.0

# Step 1: compute L2 norm of gradient
total_norm = torch.norm(grad).item()
print(f"Original gradient: {grad.tolist()}")
print(f"Gradient norm: {total_norm:.4f}")
print()

# Step 2: if norm exceeds threshold, scale proportionally
if total_norm > max_norm:
    scale = max_norm / total_norm
    clipped = grad * scale
    print(f"Norm {total_norm:.4f} > threshold {max_norm}")
    print(f"Scale factor: {max_norm}/{total_norm:.4f} = {scale:.4f}")
    print(f"After clipping: {[f'{v:.4f}' for v in clipped.tolist()]}")
    print(f"Clipped norm: {torch.norm(clipped).item():.4f} (= {max_norm})")
else:
    clipped = grad
    print(f"Norm {total_norm:.4f} <= threshold {max_norm}, no clipping needed")

print()
# PyTorch built-in implementation
grad_copy = grad.clone()
torch.nn.utils.clip_grad_norm_(grad_copy, max_norm)
print(f"PyTorch clip_grad_norm_ result: {[f'{v:.4f}' for v in grad_copy.tolist()]}")
print(f"Same as our hand calculation")
print()
print("Practical usage (in training loop):")
print("  optimizer.zero_grad()")
print("  loss.backward()")
print("  torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)")
print("  optimizer.step()")
print()
print("Almost all LLM training uses clip=1.0, a very stable default")

#### 11.2 Gradient Accumulation: Split into Steps When Memory Is Insufficient

The bottleneck of large model training is often **memory**. A 7B model with batch_size=4 can fill up an 80GB A100.

But scaling laws tell us: larger batches train better. batch_size=4 isn't enough, but what if we want 32?

**Gradient Accumulation** approach: split a large batch into several small batches, compute gradients for each, accumulate them, and update parameters once at the end.

```
Goal: batch_size = 32 (but memory only fits 4)

Step 1: small batch 1 (4 samples) -> compute gradient -> accumulate
Step 2: small batch 2 (4 samples) -> compute gradient -> accumulate
...
Step 8: small batch 8 (4 samples) -> compute gradient -> accumulate
-> After 8 accumulation steps, gradient = sum of 8 small batch gradients
-> Divide by 8 to average -> update parameters

Equivalent to using batch_size=32 all at once!
```

In [ ]:
# === Gradient Accumulation hand calculation ===
import torch

print("=== Gradient Accumulation Hand Calculation ===")
print()

# Simulate 4 small batches, each computing its own gradient
grads = [
    torch.tensor([0.5, -0.3, 0.8]),
    torch.tensor([0.2, -0.1, 0.6]),
    torch.tensor([0.7, -0.4, 0.3]),
    torch.tensor([0.3, -0.2, 0.5]),
]
accumulation_steps = len(grads)

print(f"Accumulation steps: {accumulation_steps}")
print(f"Gradient from each small batch:")
for i, g in enumerate(grads):
    print(f"  Step {i+1}: {g.tolist()}")
print()

# Accumulate
accumulated = torch.zeros_like(grads[0])
for g in grads:
    accumulated += g

# Average
averaged = accumulated / accumulation_steps

print(f"Accumulated sum: {accumulated.tolist()}")
print(f"After averaging: {averaged.tolist()}")
print()

# Compare: gradient computed from all data at once
# Assuming loss is independent per sample, large batch gradient = average of small batch gradients
big_batch_grad = torch.stack(grads).mean(dim=0)
print(f"Direct large batch: {big_batch_grad.tolist()}")
print(f"Accumulated then averaged: {averaged.tolist()}")
print(f"-> Identical")
print()
print("Training loop pattern:")
print("  for i, batch in enumerate(dataloader):")
print("      loss = model(batch) / accumulation_steps  # divide by steps")
print("      loss.backward()                          # gradients auto-accumulate")
print("      if (i + 1) % accumulation_steps == 0:")
print("          optimizer.step()                     # update parameters")
print("          optimizer.zero_grad()                # clear gradients")

#### 11.3 Warmup: "Jogging to Warm Up" at the Start of Training

When the model is first initialized, parameters are random. If we use a large learning rate (e.g., 0.01) from the start, gradient directions are chaotic, parameters get "pulled all over the place," and loss may explode.

**Warmup approach**: for the first N steps (usually 5% of total steps), linearly increase the learning rate from 0 to the target value. Let the model "probe" with small steps first, find roughly the right direction, then accelerate.

```
  LR
  |        /----------------------  <- normal training
  |      /
  |    /
  |  /
  |/  <- Warmup phase (first 5% of steps)
  +-------------------------------> Step
```

Real-world intuition for warmup: starting a car in winter, idle for 30 seconds before driving -- flooring the gas immediately would damage the engine.

In [ ]:
# === Warmup hand calculation + visualization ===
import matplotlib.pyplot as plt

import math

print("=== Warmup Hand Calculation ===")
print()

total_steps = 1000
warmup_steps = 50   # first 50 steps are warmup
max_lr = 0.01

def warmup_lr(step, warmup_steps, max_lr, total_steps):
    """Linear warmup + cosine decay"""
    if step < warmup_steps:
        return max_lr * step / warmup_steps
    else:
        progress = (step - warmup_steps) / (total_steps - warmup_steps)
        return max_lr * 0.5 * (1 + math.cos(math.pi * progress))

# Key checkpoint LRs
check_points = [0, 10, 25, 50, 100, 500, 900, 999]
print(f"Total steps: {total_steps}, Warmup steps: {warmup_steps}, Max LR: {max_lr}")
print()
print(f"{'Step':>6s}  {'LR':>12s}  {'Phase'}")
print('-' * 40)
for step in check_points:
    lr = warmup_lr(step, warmup_steps, max_lr, total_steps)
    phase = 'Warmup' if step < warmup_steps else 'Training'
    print(f"{step:>6d}  {lr:>12.6f}  {phase}")

print()
print("Key observations:")
print(f"  Step 0: LR = 0 (no movement, waiting for warmup)")
print(f"  Step 10: LR = {warmup_lr(10, warmup_steps, max_lr, total_steps):.6f} (gradually increasing)")
print(f"  Step 50: LR = {warmup_lr(50, warmup_steps, max_lr, total_steps):.6f} (reached max, warmup complete)")
print(f"  Step 500: LR = {warmup_lr(500, warmup_steps, max_lr, total_steps):.6f} (cosine decay in progress)")

# Visualization
steps = list(range(total_steps))
lrs = [warmup_lr(s, warmup_steps, max_lr, total_steps) for s in steps]

plt.figure(figsize=(10, 3))
plt.plot(steps, lrs, linewidth=1.5)
plt.axvspan(0, warmup_steps, alpha=0.2, color='orange', label='Warmup')
plt.axvspan(warmup_steps, total_steps, alpha=0.05, color='blue', label='Cosine Decay')
plt.xlabel('Step')
plt.ylabel('Learning Rate')
plt.title('Warmup + Cosine Decay (standard LLM training schedule)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
print("Orange region = Warmup (linear increase), Blue region = normal training (cosine decay)")
print("Part 12's WSD scheduler replaces cosine with 'constant' -- more flexible")

**Appendix: MUON Optimizer -- Replacing Per-Element Learning Rates with Matrix Orthogonalization**

AdamW is the most commonly used optimizer today, but it has an implicit assumption: each parameter is independent. For Transformer's 2D weight matrices (Attention's Q, K, V projections, FFN's up/down projections), this assumption doesn't hold.

A weight matrix $W \in \mathbb{R}^{m \times n}$ carries structural information -- its singular value distribution, condition number -- that significantly affects gradient update effectiveness. AdamW adjusts learning rates independently per element, which changes the matrix's spectral properties (singular value spectrum), weakening the matrix's inherent structure.

Public sources indicate that new models like DeepSeek V4 are starting to adopt the MUON (MomentUm Orthogonalized by Newton-Schulz) approach. It takes a different approach: for 2D weight matrices in hidden layers, instead of per-element learning rate adjustment, it uses Newton-Schulz iteration to orthogonalize the gradient matrix, then applies unified momentum updates.

**Core intuition**: Think of the gradient matrix G as a set of "update directions." These directions may have strong correlations -- some directions are repeatedly reinforced while others are ignored. Newton-Schulz orthogonalization reorganizes these directions to be mutually orthogonal, so each update direction contributes independently without interfering with others.

The Newton-Schulz iteration computes $G (G^T G)^{-1/2}$, transforming the gradient matrix into a semi-orthogonal matrix (column vectors are mutually orthogonal with unit norm). Directly computing the inverse matrix square root is expensive, so iterative approximation is used:

```python
X = G / ||G||_F          # normalize first
for _ in range(5):        # 5 iterations to converge
    A = X @ X.T
    X = 1.5 * X - 0.5 * A @ X
# X is now the approximately orthogonalized gradient
```

This iteration only involves matrix multiplication, which is very fast on GPU -- additional overhead is only 0.5%~0.7% FLOP.

Comparison with AdamW:

|  | AdamW | MUON |
|:---|:---|:---|
| Processing approach | Per-element adaptive learning rate | Matrix-level orthogonalization + momentum |
| Optimizer state | 2x parameter count (m, v) | 1x parameter count (only momentum) |
| Applicable parameters | All parameters | Typically used for hidden 2D weight matrices |
| 1D parameters | Uses AdamW | Still uses AdamW (bias, Norm, etc.) |

In practice, a mixed strategy is typical: hidden layer 2D weight matrices use MUON, while Embedding, lm_head, Norm, and bias parameters still use AdamW. This is similar to setting different optimizers for different parameter groups in training code.

References: [PyTorch/DeepSpeed MUON introduction](https://pytorch.org/blog/using-muon-optimizer-with-deepspeed/), [Moonlight/MUON paper](https://arxiv.org/abs/2412.13663).

Below is a minimal example demonstrating the effect of Newton-Schulz orthogonalization.

In [ ]:
# === MUON's Newton-Schulz orthogonalization minimal demo ===
import torch

print("=== Newton-Schulz Orthogonalization: turning gradient matrix into orthogonal matrix ===")
print()

torch.manual_seed(42)

# Simulate a 2D weight matrix gradient (e.g., W_q: d_model=64, d_model=64)
m, n = 64, 64
G = torch.randn(m, n) * 2.0  # simulated gradient, columns have correlation
# Artificially create column correlation: make the second half columns a linear combination of the first half
G[:, n//2:] = G[:, :n//2] @ torch.randn(n//2, n//2) * 0.3 + G[:, n//2:]

print(f"Original gradient matrix G: {m}x{n}")
print(f"  Frobenius norm: {torch.norm(G, 'fro'):.2f}")
# Check column correlation (using off-diagonal elements of Gram matrix)
gram = G.T @ G
off_diag = gram - torch.diag(torch.diag(gram))
print(f"  Column correlation (Gram off-diagonal norm): {torch.norm(off_diag, 'fro'):.2f}")
print()

# Newton-Schulz iteration
X = G / torch.norm(G, 'fro')  # Step 0: normalize
print("Newton-Schulz iteration process:")
for step in range(5):
    A = X @ X.T
    X_new = 1.5 * X - 0.5 * A @ X
    # Check convergence toward orthogonality
    gram_X = X_new.T @ X_new
    off_diag_X = gram_X - torch.diag(torch.diag(gram_X))
    error = torch.norm(off_diag_X, 'fro').item()
    print(f"  Step {step+1}: off-diagonal norm = {error:.6f}")
    X = X_new

print()
print("Orthogonalized gradient matrix X:")
# Verify orthogonality
gram_final = X.T @ X
off_diag_final = torch.norm(gram_final - torch.diag(torch.diag(gram_final)), 'fro')
print(f"  Column correlation (Gram off-diagonal norm): {off_diag_final:.6f}")
# Verify each column has norm close to 1
col_norms = torch.norm(X, dim=0)
print(f"  Per-column norm range: [{col_norms.min().item():.4f}, {col_norms.max().item():.4f}]")

print()
print("Key observations:")
print("  - Original gradient has significant inter-column correlation (large off-diagonal norm)")
print("  - After 5 Newton-Schulz iterations, columns are nearly orthogonal (off-diagonal -> 0)")
print("  - Each column's norm is close to 1, i.e., X is a semi-orthogonal matrix")
print("  - MUON uses this orthogonalized X instead of original gradient G for momentum updates")
print("  - Intuitive understanding: removes correlation between gradient directions, each direction contributes independently")

## 12. Multi-Token Prediction -- Predicting Multiple Future Tokens at Once (DeepSeek-V3)

In standard training, each position only predicts the immediately next token. The hidden state at position t passes through the LM Head to output a prediction for token_{t+1}, which is compared against target at position t+1 using cross-entropy. All positions do this, and the loss is averaged -- this is the training method we've been using throughout.

Multi-Token Prediction (MTP) extends this rule: the hidden state at position t must predict not only token_{t+1}, but also token_{t+2}, token_{t+3}, ... up to token_{t+N}.

```
Standard:     hidden_t -> Head -> P(token_{t+1})
MTP (N=3):    hidden_t -> Head_1 -> P(token_{t+1})
                        -> Head_2 -> P(token_{t+2})
                        -> Head_3 -> P(token_{t+3})
```

In implementation, N parallel output heads are attached after the last Transformer layer. Each head is an independent Linear layer (in real models, each head is a small Transformer block). During training, Head_1 compares against target[:,1:] (standard next-token), Head_2 compares against target[:,2:] (token two steps ahead), and so on. Each head's loss is computed as cross-entropy independently, and the total loss is the average across all heads.

The direct effect is denser training signals. In standard training, token_{t+k} is supervised only once by the hidden state at position t+k-1. In MTP, it's also predicted from t+k-2, t+k-3, etc. by hidden states from further away. Each token receives multiple supervision signals from different distances.

During standard autoregressive generation, you can keep only Head_1 (standard next-token prediction) and discard the auxiliary heads; the benefit of MTP comes mainly from the denser supervision during training. However, the DeepSeek-V3 paper and open-source implementation also show that MTP modules can be used as draft modules for speculative decoding: auxiliary heads first guess multiple future tokens, then the main model verifies them, accelerating generation in suitable inference frameworks. So a more accurate statement is: **MTP can be discarded during standard generation, or reused in speculative decoding**. Reference: [DeepSeek-V3 GitHub](https://github.com/deepseek-ai/DeepSeek-V3), [DeepSeek-V3 Technical Report](https://arxiv.org/abs/2412.19437).

In [ ]:
import torch.nn as nn
import torch.nn.functional as F

class MultiHeadLM(nn.Module):
    """
    Multi-token prediction output heads

    Attaches N output heads after the main model, each predicting
    future tokens at distance 1~N.
    In real models (DeepSeek-V3), each head is a small Transformer block;
    here we use a single Linear layer to demonstrate the principle.

    Args:
        d_model: hidden dimension
        vocab_size: vocabulary size
        num_heads: number of prediction heads (including main head)
    """
    def __init__(self, d_model, vocab_size, num_heads=4):
        super().__init__()
        self.num_heads = num_heads

        self.heads = nn.ModuleList([
            nn.Linear(d_model, vocab_size, bias=False)
            for _ in range(num_heads)
        ])

    def forward(self, hidden_states):
        """
        hidden_states: [batch, seq_len, d_model]

        Returns: list of logits
              head[i] predicts token_{pos + i + 1}
        """
        return [head(hidden_states) for head in self.heads]

def compute_mtp_loss(logits_list, target_ids, ignore_index=-100):
    """
    Compute total MTP loss

    head_i predicts the (i+1)-th future token, needs to align with targets
    - head_0 compares against target[:, 1:]   (next token)
    - head_1 compares against target[:, 2:]   (token two steps ahead)
    - head_2 compares against target[:, 3:]   (token three steps ahead)
    """
    total_loss = 0.0
    for i, logits in enumerate(logits_list):
        shift = i + 1
        # Remove last 'shift' positions (no corresponding target)
        logits_trimmed = logits[:, :-shift, :]
        targets = target_ids[:, shift:]

        logits_flat = logits_trimmed.reshape(-1, logits.shape[-1])
        targets_flat = targets.reshape(-1)

        total_loss += F.cross_entropy(logits_flat, targets_flat,
                                      ignore_index=ignore_index)
    return total_loss / len(logits_list)

print("Multi-Token Prediction components defined!")
print("Key: N independent output heads, each predicting future tokens at different distances; only the main head is kept during inference")

In [ ]:
# Demo: MTP loss computation
import torch
import torch.nn.functional as F

torch.manual_seed(42)

V = 20
B, S, D = 2, 8, 32

# Simulate hidden states from the main model and correct target ids
hidden = torch.randn(B, S, D)
targets = torch.randint(0, V, (B, S))

mtp = MultiHeadLM(D, V, num_heads=4)
logits_list = mtp(hidden)

print("=== Multi-Token Prediction Loss Demo ===")
print(f"Input shape: batch={B}, seq_len={S}, d_model={D}")
print(f"Number of prediction heads: {len(logits_list)}")
print()

# Compare: standard single-head training vs MTP
print("Standard training (single head):")
single_loss = F.cross_entropy(
    logits_list[0][:, :-1, :].reshape(-1, V),
    targets[:, 1:].reshape(-1)
)
print(f"  Only Head 0 predicts t+1, loss = {single_loss.item():.4f}")
print(f"  Each token is supervised 1 time")

print()
print("MTP training (4 heads):")
for i, logits in enumerate(logits_list):
    shift = i + 1
    effective = S - shift
    head_loss = F.cross_entropy(
        logits[:, :-shift, :].reshape(-1, V),
        targets[:, shift:].reshape(-1)
    )
    print(f"  Head {i} (predicts t+{shift}): effective positions={effective}, loss={head_loss.item():.4f}")

total = compute_mtp_loss(logits_list, targets)
print(f"  Total MTP loss (average): {total.item():.4f}")

print()
print("Key observations:")
print("1. Head 0 is standard next-token prediction -- identical to single-head training")
print("2. Heads 1~3 predict further tokens, with fewer effective positions per head (no labels at sequence end)")
print("3. The same hidden state produces 4 supervision signals -> higher training information density")
print("4. During inference, only Head 0 is kept; Heads 1~3 are all discarded, inference speed unaffected")

## Summary

Confirm the following understanding:

1. [ ] Training input = complete sentence with last token removed; labels = complete sentence with first token removed (shifted right by one)
2. [ ] All token positions make predictions simultaneously, each computing cross-entropy loss independently
3. [ ] Total loss = average of loss across all valid positions
4. [ ] **This is token-level training, but all tokens are computed in parallel** (not sentence-level, not sequential tokens)
5. [ ] Training uses teacher forcing (ground truth labels); inference can only generate sequentially
6. [ ] PAD positions are excluded using `ignore_index`, not contributing to loss

**This knowledge applies not just to our mini GPT, but to all autoregressive language models (GPT-2/3/4, LLaMA, Qwen, ...).**

The above is the standard training framework. In practice, Multi-Token Prediction (MTP) can also be used
to make each position predict N future tokens simultaneously, increasing training signal density (see Section 12).

-> Final Part: the model is trained, how do we make it "speak"?

## Exercises

**Exercise 1: Hand-calculate Cross Entropy**

Given logits `[2.0, 1.0, 0.0]` and target class 0, compute the softmax probabilities and cross entropy loss by hand. Verify your answer with `torch.nn.functional.cross_entropy`.

**Exercise 2: Design a loss mask**

Construct a simplified conversation with system, user, and assistant token spans. Only the assistant answer should contribute to loss; system and user positions should have mask value 0. Print `input_ids`, `labels`, and `loss_mask` to confirm that only answer tokens are trained.

**Exercise 3: MTP weighting experiment**

Set Multi-Token Prediction weights for step 1/2/3 to `[1.0, 0.5, 0.25]`, then change them to `[1.0, 1.0, 1.0]`. Compare total loss and explain why farther-future token prediction is usually harder.